# USGS Earthquake Dataset Header Dictionary

This schema represents standard earthquake event data provided by the **USGS (United States Geological Survey)**, annotated with recommendations for **time-series and spatiotemporal model training** (e.g., forecasting, sequence models, seismic hazard prediction).

## Data Summary & ML Training Suitability
| Header | Sample Value | Data Type | Unit / Interpretation | Description | Usable for Time-Series ML? |
|----------|----------|----------|----------|----------|----------|
| `time` | `2025-12-07T02:43:07.342Z` | ISO UTC Timestamp | Exact earthquake origin time in UTC. | Time when rupture began. | **Yes (Primary Time Index)** |
| `latitude` | `29.8698` | Float | 29.8698° North. Range: -90° to +90°. | North-South geographic position. | **Yes (Core Spatial Feature)** |
| `longitude` | `80.9384` | Float | 80.9384° East. Range: -180° to +180°. | East-West geographic position. | **Yes (Core Spatial Feature)** |
| `depth` | `43.404` | Float (km) | Earthquake started 43.404 km below the surface. | Hypocenter depth. | **Yes (Core Spatial Feature)** |
| `mag` | `4.4` | Float | Magnitude 4.4 earthquake. +1 magnitude ≈ 31.6× more energy. | Earthquake size/strength. | **Yes (Target Variable / Key Feature)** |
| `magType` | `mb` | String | Body-Wave Magnitude scale. Other values: `Mw`, `ML`, `Ms`. | Method used to compute magnitude. | **Optional (Categorical)** |
| `nst` | `21` | Integer | 21 seismic stations contributed to location estimation. | Station count used for location. | **Optional (Quality Weighting)** |
| `gap` | `166` | Float (degrees) | Largest angular gap between stations around the epicenter. Smaller is better. | Network coverage quality metric. | **Optional (Quality Weighting)** |
| `dmin` | `8.856` | Float (degrees) | Nearest station was ~983 km away (1° ≈ 111 km). | Distance to closest station. | **Optional (Quality Weighting)** |
| `rms` | `0.94` | Float (seconds) | Average travel-time prediction error of 0.94 s. Lower is better. | Model fit quality. | **Optional (Quality Weighting)** |
| `horizontalError` | `13.18` | Float (km) | Epicenter location uncertainty ≈ 13.18 km. | Horizontal location confidence. | **Optional (Uncertainty Weighting)** |
| `depthError` | `9.35` | Float (km) | Depth estimate uncertainty ≈ 9.35 km. | Depth confidence metric. | **Optional (Uncertainty Weighting)** |
| `magError` | `0.144` | Float | Magnitude uncertainty ±0.144. | Magnitude confidence metric. | **Optional (Uncertainty Weighting)** |
| `magNst` | `14` | Integer | 14 stations used specifically for magnitude calculation. | Magnitude measurement quality. | **Optional (Quality Weighting)** |
| `type` | `earthquake` | String | Event classification. | Earthquake, explosion, quarry blast, etc. | **Filter Only** |
| `status` | `reviewed` | String | Reviewed by a human seismologist. | Verification status. | **Filter Only** |
| `net` | `us` | String | USGS reporting network identifier. | Administrative metadata. | **No (Drop)** |
| `id` | `us7000rkd1` | String | Unique event identifier only. | Arbitrary database ID. | **No (Drop)** |
| `updated` | `2025-12-26T01:20:25.040Z` | ISO UTC Timestamp | Record modification timestamp. | Last update time. | **No (Drop — Leakage Risk)** |
| `place` | `37 km E of Dārchulā...` | String | Human-readable location description. | Textual location. | **No (Drop)** |
| `locationSource` | `us` | String | Agency that computed location. | Administrative metadata. | **No (Drop)** |
| `magSource` | `us` | String | Agency that computed magnitude. | Administrative metadata. | **No (Drop)** |

In [1]:
# CONSTANTS
ORGINAL_DATA_PATH = 'data/nepal_eq_90_26_raw.csv'
FILTERED_DATA_PATH = 'data/nepal_eq_90_26_filtered.csv'
INITIAL_EXTRACTION_DATA_PATH = 'data/nepal_eq_90_26_initial_extraction.csv'

def INITIAL_EXTRACTION_DATA_PATH_TEMPLATE(version: str) -> str:
	return f'data/nepal_eq_90_26_initial_extraction_v{version}.csv'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df_original = pd.read_csv(ORGINAL_DATA_PATH)


# Filtering the dataset to include only relevant features
- Removing all the features that are not relevant to the analysis. The relevant features are mentioned 
in the Features class below. The filtered dataset is saved as 'data/nepal_eq_90_26_filtered.csv'.
and the rest columns are dropped since they are metadata about earthquate and the source of the data.

In [3]:
from enum import Enum
class Features(Enum):
	TIME = 'time'
	LATITUDE = 'latitude'
	LONGITUDE = 'longitude'
	DEPTH = 'depth'
	MAGNITUDE = 'mag'
	MAGNITUDE_TYPE = 'magType'
	NSTATIONS = 'nst'
	GAP = 'gap'
	DMIN = 'dmin'
	RMS = 'rms'
	HORIZONTAL_ERROR = 'horizontalError'
	DEPTH_ERROR = 'depthError'
	MAGNITUDE_ERROR = 'magError'
	MAGNITUDE_NSTATIONS = 'magNst'	

	DISTRICT = 'district'

df_filtered = df_original[[f.value for f in Features if f != Features.DISTRICT]]
df_filtered.head()

df_filtered.to_csv(FILTERED_DATA_PATH, index=False)

## Radius Finding V1

In [9]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.special import lambertw

def compute_felt_radius_v1(
	magnitude: float,
	depth_km: float,
	target_mmi: float = 2.5,
	region: str = "active"
) -> float:
	"""
	Computes surface felt radius R_felt (km): the epicentral distance at which
	ground shaking is predicted to reach `target_mmi`, using a regional
	tectonic attenuation model.
	"""

	# 1. Master Configuration Dictionaries
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.139, 1.093, 1.150, 0.00197),   # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}

	
	# 2. Extract settings with safety defaults
	c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])

	# 3. Finite-fault saturation term
	R0 = 10 ** (-0.281 + 0.251 * magnitude)

	# 4. Lambert W setup for 3D effective distance
	a = c3 / c4
	b = (c1 + c2 * magnitude - target_mmi) / c4

	ln10 = np.log(10)
	z = (ln10 / a) * (10.0 ** (b / a))
	R_eff = (a / ln10) * lambertw(z).real

	# 5. Pythagorean depth / finite-fault correction to surface distance
	rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

	if rad_sq <= 0:
		return 0.0

	return float(np.sqrt(rad_sq))


# --- Executing Test ---
m, h, target = 7.8, 8.22, 3
radius = compute_felt_radius_v1(m, h, target_mmi=target, region="active")
print(f"--- MMI\u2265{target} Felt Radius for Mag {m} Event (Depth: {h}km) ---")
print(f"-> Region: active -> Radius: {radius:.1f} km\n")

--- MMI≥3 Felt Radius for Mag 7.8 Event (Depth: 8.22km) ---
-> Region: active -> Radius: 672.1 km



In [ ]:
@dataclass(frozen=True)
class FeltRadiusConfigV1:
	"""
	Configures the parameters for felt radius calculation.
	Allows customization of input/output column names and region type.
	`i_` prefix denotes input columns, and `o_` prefix denotes output columns.
	"""
	i_region: str = "active"
	i_time_column: str = "time"
	i_mag_column: str = "mag"
	i_depth_column: str = "depth"
	o_felt_radius_column: str = "felt_radius_km"
	o_felt_area_column: str = "felt_area_sqkm"
	o_max_nearfield_mmi_column: str = "max_nearfield_mmi"


class FeltRadiusCalculatorV1:
	coefficient_configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.139, 1.093, 1.150, 0.00197),   # Central & Eastern US
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
	}
	
	supported_tectonic_regions = set(coefficient_configurations.keys())

	@staticmethod
	def extract_seismic_features(
		df: pd.DataFrame, 
		felt_mmi: float = 2.5, 
		config: FeltRadiusConfigV1 = FeltRadiusConfigV1()
	) -> pd.DataFrame:
		"""
		Extracts physical felt-vibration features from earthquake catalog data
		incorporating regional tectonic attenuation.
		"""
		data = df.copy()
		
		# 1. Fetch and validate regional configurations
		region = config.i_region
		
		if region in FeltRadiusCalculatorV1.coefficient_configurations:
			c1, c2, c3, c4 = FeltRadiusCalculatorV1.coefficient_configurations[region]
		else:
			raise ValueError(f"Unknown region profile: {region}, supported: {FeltRadiusCalculatorV1.supported_tectonic_regions}")
			
		M = data[config.i_mag_column].values
		h = data[config.i_depth_column].fillna(10.0).values  # Default to 10km depth if missing
		
		# 2. Near-field finite fault saturation term R0(M)
		R0 = 10 ** (-0.281 + 0.251 * M)
		
		# 3. Transformed Lambert W parameters
		a = c3 / c4
		b = (c1 + c2 * M - felt_mmi) / c4
		
		# 4. Vectorized Lambert W evaluation for R_eff across DataFrame arrays
		ln10 = np.log(10)
		z = (ln10 / a) * (10.0 ** (b / a))
		R_eff = (a / ln10) * lambertw(z).real
		
		# 5. Surface felt radius accounting for depth and saturation
		rad_sq = R_eff**2 - R0**2 - h**2
		r_felt = np.sqrt(np.maximum(rad_sq, 0.0))
		
		# 6. Append physical feature columns
		data[config.o_felt_radius_column] = r_felt
		data[config.o_felt_area_column] = np.pi * (r_felt ** 2)
		
		# 7. Peak intensity directly above hypocenter (R_jb = 0)
		R_eff_epicenter = np.sqrt(R0**2 + h**2)
		base_max_mmi = c1 + c2 * M - c3 * np.log10(R_eff_epicenter) - c4 * R_eff_epicenter
		data[config.o_max_nearfield_mmi_column] = base_max_mmi
		
		return data


config = FeltRadiusConfigV1(
	i_region="active",
	i_time_column="time",
	i_mag_column="mag",
	i_depth_column="depth",
	o_felt_radius_column="felt_radius_km",
	o_felt_area_column="felt_area_sqkm",
	o_max_nearfield_mmi_column="max_nearfield_mmi"
)
mmi = 4.0
df_extracted_1 = FeltRadiusCalculatorV1.extract_seismic_features(
	df_filtered, 
	felt_mmi=mmi, 
	config=config
)
df_extracted_1.to_csv(INITIAL_EXTRACTION_DATA_PATH_TEMPLATE("1"), index=False)

NameError: name 'df_filtered' is not defined

In [10]:
earthquake_catalog = [
    {
        "name": "2015 Gorkha (Nepal)",
        "magnitude": 7.8,
        "depth_km": 15.0,
        "region": "active",
        "observed_felt_radius_km": "~800 - 1000",
        "notes": "Felt widely across Northern India, Tibet, and Bangladesh"
    },
    {
        "name": "2011 Mineral (Virginia, USA)",
        "magnitude": 5.8,
        "depth_km": 6.0,
        "region": "stable_cena",
        "observed_felt_radius_km": "~800 - 1000",
        "notes": "Very low attenuation in CENA; felt in NYC, Toronto & Atlanta"
    },
    {
        "name": "2011 Tohoku (Japan)",
        "magnitude": 9.1,
        "depth_km": 29.0,
        "region": "subduction",
        "observed_felt_radius_km": "> 1200",
        "notes": "Megathrust event; shaking felt throughout Honshu and eastern China"
    },
    {
        "name": "2023 Kahramanmaraş (Turkey)",
        "magnitude": 7.8,
        "depth_km": 10.0,
        "region": "strike_slip",
        "observed_felt_radius_km": "~700 - 900",
        "notes": "Major strike-slip rupture; felt in Syria, Cyprus, Egypt & Iraq"
    },
    {
        "name": "1989 Loma Prieta (California)",
        "magnitude": 6.9,
        "depth_km": 19.0,
        "region": "strike_slip",
        "observed_felt_radius_km": "~350 - 450",
        "notes": "San Andreas strike-slip; higher active crust attenuation"
    },
    {
        "name": "2021 Fagradalsfjall (Iceland)",
        "magnitude": 5.7,
        "depth_km": 5.0,
        "region": "volcanic",
        "observed_felt_radius_km": "~100 - 150",
        "notes": "High thermal attenuation in rift zone limits long-range shaking"
    },
    {
        "name": "2010 Christchurch (New Zealand)",
        "magnitude": 6.2,
        "depth_km": 5.0,
        "region": "active",
        "observed_felt_radius_km": "~250 - 350",
        "notes": "Shallow active crustal event felt across South Island"
    }
]

# Run validation across catalog
results = []
TARGET_MMI = 2.5  # Typical human perception threshold (MMI II-III)

for eq in earthquake_catalog:
    predicted_radius = compute_felt_radius_v1(
        magnitude=eq["magnitude"],
        depth_km=eq["depth_km"],
        target_mmi=TARGET_MMI,
        region=eq["region"]
    )
    results.append({
        "Earthquake Event": eq["name"],
        "Mag (Mw)": eq["magnitude"],
        "Depth (km)": eq["depth_km"],
        "Region": eq["region"],
        "Predicted Radius (km)": round(predicted_radius, 1),
        "Observed USGS Felt Radius (km)": eq["observed_felt_radius_km"]
    })

df_validation = pd.DataFrame(results)

print(f"=== EARTHQUAKE FELT RADIUS VALIDATION (Target MMI = {TARGET_MMI}) ===")
print(df_validation.to_string(index=False))

=== EARTHQUAKE FELT RADIUS VALIDATION (Target MMI = 2.5) ===
               Earthquake Event  Mag (Mw)  Depth (km)      Region  Predicted Radius (km) Observed USGS Felt Radius (km)
            2015 Gorkha (Nepal)       7.8        15.0      active                  833.2                    ~800 - 1000
   2011 Mineral (Virginia, USA)       5.8         6.0 stable_cena                  824.5                    ~800 - 1000
            2011 Tohoku (Japan)       9.1        29.0  subduction                 1629.9                         > 1200
    2023 Kahramanmaraş (Turkey)       7.8        10.0 strike_slip                  604.9                     ~700 - 900
  1989 Loma Prieta (California)       6.9        19.0 strike_slip                  392.5                     ~350 - 450
  2021 Fagradalsfjall (Iceland)       5.7         5.0    volcanic                  122.0                     ~100 - 150
2010 Christchurch (New Zealand)       6.2         5.0      active                  359.8           